# Fed-KAN on Colab GPU: FC vs B-spline KAN vs Taylor-KAN

FedAvg across LargeST districts, one district per client. Same model and training budget for every head; only the prediction head changes.

Before running: Runtime > Change runtime type > T4 GPU. Expects `MyDrive/fedkan/fedkan_code_v2.zip` and `MyDrive/fedkan/data_real/district_*`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# run settings
ROUNDS = 5
RESAMPLE = 3        # 3 = 15-min data, same as Phase 1
BATCH = 64
MAX_BATCHES = 0     # 0 = full epoch per client
TAYLOR_ORDER = 2
GRID_SIZE = 8       # Phase 1 B-spline config
SPLINE_ORDER = 3
HEADS = ['taylor', 'bspline', 'fc']

DRIVE = '/content/drive/MyDrive/fedkan'
WORK = '/content/fedkan'

In [ ]:
# copy code and data off Drive onto the local disk (much faster to read)
import os, shutil, zipfile
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
with zipfile.ZipFile(f'{DRIVE}/fedkan_code_v2.zip') as z:
    z.extractall(WORK)
shutil.copytree(f'{DRIVE}/data_real', f'{WORK}/phase2/data_real')
print(sorted(os.listdir(f'{WORK}/phase2/data_real')))

In [ ]:
%cd /content/fedkan/phase2
os.makedirs(f'{DRIVE}/results', exist_ok=True)
COMMON = (f'--data data_real --rounds {ROUNDS} --resample {RESAMPLE} --batch-size {BATCH} '
          f'--max-batches {MAX_BATCHES} --taylor-order {TAYLOR_ORDER} '
          f'--grid-size {GRID_SIZE} --spline-order {SPLINE_ORDER}')

In [ ]:
# quick smoke test of every head (a few batches) so a bug shows up in seconds, not an hour in
for head in HEADS:
    !python run_federated_local.py --head {head} --data data_real --rounds 1 --resample {RESAMPLE} --batch-size {BATCH} --max-batches 3 --max-val-batches 3

In [ ]:
# full runs, one head at a time; each result is copied to Drive as soon as it finishes
for head in HEADS:
    !python run_federated_local.py --head {head} {COMMON} --out results/colab_{head}.json
    shutil.copy(f'results/colab_{head}.json', f'{DRIVE}/results/colab_{head}.json')

In [ ]:
# compare the heads and save the table and chart to Drive
import json
import matplotlib.pyplot as plt

runs = {}
for head in HEADS:
    path = f'results/colab_{head}.json'
    if os.path.exists(path):
        with open(path) as f:
            runs[head] = json.load(f)

lines = [f"{'head':10}{'head params':>13}{'MAE':>9}{'RMSE':>9}{'R2':>8}{'min':>8}"]
for head, r in runs.items():
    last = r['rounds'][-1]
    lines.append(f"{head:10}{r['head_params']:>13,}{last['mae']:>9.3f}{last['rmse']:>9.3f}{last['r2']:>8.3f}{last['minutes']:>8.1f}")
table = '\n'.join(lines)
print(table)
with open(f'{DRIVE}/results/summary.txt', 'w') as f:
    f.write(table + '\n')

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for head, r in runs.items():
    rounds = [x['round'] for x in r['rounds']]
    ax[0].plot(rounds, [x['mae'] for x in r['rounds']], marker='o', label=head)
    ax[1].plot(rounds, [x['r2'] for x in r['rounds']], marker='o', label=head)
ax[0].set_title('Validation MAE')
ax[1].set_title('Validation R2')
for a in ax:
    a.set_xlabel('round')
    a.legend()
plt.tight_layout()
plt.savefig(f'{DRIVE}/results/heads_comparison.png', dpi=150)
plt.show()
print('saved to', f'{DRIVE}/results')